# Distributed Training

### Load Data

In [68]:
import pandas as pd

In [69]:
DATASET_LOC = "https://raw.githubusercontent.com/GokuMohandas/Made-With-ML/main/datasets/dataset.csv"
train_df = pd.read_csv(DATASET_LOC)
train_df.head()

,id,created_on,title,description,tag
0,6,2020-02-20 06:43:18,Comparison between YOLO and RCNN on real world...,Bringing theory to experiment is cool. We can ...,computer-vision
1,7,2020-02-20 06:47:21,"Show, Infer & Tell: Contextual Inference for C...",The beauty of the work lies in the way it arch...,computer-vision
2,9,2020-02-24 16:24:45,Awesome Graph Classification,"A collection of important graph embedding, cla...",other
3,15,2020-02-28 23:55:26,Awesome Monte Carlo Tree Search,A curated list of Monte Carlo tree search pape...,other
4,25,2020-03-07 23:04:31,AttentionWalk,"A PyTorch Implementation of ""Watch Your Step: ...",other


In [70]:
train_df.shape

(764, 5)

In [71]:
# Unique Labels
tags = train_df["tag"].unique().tolist()
tags

['computer-vision', 'other', 'natural-language-processing', 'mlops']

In [72]:
HOLDOUT_LOC = "https://raw.githubusercontent.com/GokuMohandas/Made-With-ML/main/datasets/holdout.csv"
test_df = pd.read_csv(HOLDOUT_LOC)

### Utilities


In [73]:
import matplotlib.pyplot as plt
import json
from collections import Counter
import seaborn as sns; sns.set_theme()
from sklearn.metrics import precision_recall_fscore_support
import time
from tqdm import tqdm
import torch
from transformers import pipeline, Trainer, TrainingArguments, DistilBertTokenizer, DistilBertForSequenceClassification

In [74]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [75]:
class_to_index = {label: i for i, label in enumerate(tags)}
index_to_class = {i : label for i, label in enumerate(tags)}

In [76]:
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")
model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english",
                                                            id2label = index_to_class,
                                                            label2id = class_to_index,
                                                            ignore_mismatched_sizes=True)

inputs = tokenizer(train_df["title"][1] + " " + train_df["description"][1], return_tensors="pt", padding="longest")
with torch.no_grad():
    logits = model(**inputs).logits
predicted_class_id = logits.argmax().item()
model.config.id2label[predicted_class_id]



/home/krekken/madewithml/venv/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased-finetuned-sst-2-english and are newly initialized because the shapes did not match:
- classifier.weight: found shape torch.Size([2, 768]) in the checkpoint and torch.Size([4, 768]) in the model instantiated
- classifier.bias: found shape torch.Size([2]) in the checkpoint and torch.Size([4]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


'computer-vision'

In [77]:
def get_tag(model, text, tokenizer):
    inputs = tokenizer(text, return_tensors="pt")

    with torch.no_grad():
        logits = model(**inputs).logits

    predicted_class_id = logits.argmax().item()
    return model.config.id2label[predicted_class_id]    

In [78]:
text = train_df["title"][0] + " " + train_df["description"][0]
get_tag(model, text, tokenizer)


'other'

In [79]:
# list of dicts with (title, description)
samples = test_df[["title", "description"]].to_dict(orient="records")[:3]
samples

[{'title': 'Diffusion to Vector',
  'description': 'Reference implementation of Diffusion2Vec (Complenet 2018) built on Gensim and NetworkX. '},
 {'title': 'Graph Wavelet Neural Network',
  'description': 'A PyTorch implementation of "Graph Wavelet Neural Network" (ICLR 2019) '},
 {'title': 'Capsule Graph Neural Network',
  'description': 'A PyTorch implementation of "Capsule Graph Neural Network" (ICLR 2019).'}]

In [80]:
# get a list of predictions

def get_predictions(model, text, tokenizer):
    y_pred = []
    for item in tqdm(text):
        input = str(item)
        predicted_tag = get_tag(model, input, tokenizer)

        while predicted_tag is None:
            time.sleep(30)
            predicted_tag = get_tag(model, input, tokenizer)

        y_pred.append(predicted_tag)

    return y_pred

In [81]:
get_predictions(model, samples, tokenizer)

100%|██████████| 3/3 [00:00<00:00, 39.25it/s]


['natural-language-processing',
 'natural-language-processing',
 'natural-language-processing']

In [82]:
test_df.head(3)

,id,created_on,title,description,tag
0,19,2020-03-03 13:54:31,Diffusion to Vector,Reference implementation of Diffusion2Vec (Com...,other
1,26,2020-03-07 23:11:58,Graph Wavelet Neural Network,"A PyTorch implementation of ""Graph Wavelet Neu...",other
2,44,2020-03-08 00:32:58,Capsule Graph Neural Network,"A PyTorch implementation of ""Capsule Graph Neu...",other


### Setup

In [83]:
import os
import random
from ray.data.preprocessor import Preprocessor
import numpy as np
import ray

In [84]:
def set_seeds(seed = 42):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    eval("setattr(torch.backends.cudnn, 'deterministic', True)")
    eval("setattr(torch.backends.cudnn, 'benchmark, False')")
    os.environ["PYTHONHASHSEED"] = str(seed)


In [85]:
def load_data(num_samples = None):
    ds = ray.data.read_csv(DATASET_LOC)
    ds = ds.random_shuffle(seed=1234)
    ds = ray.data.from_item(ds.take(num_samples)) if num_samples else ds
    return ds

In [86]:
sample_ds = load_data()


2026-02-26 20:39:08,300	INFO read_api.py:406 -- To satisfy the requested parallelism of 40, each read task output is split into 40 smaller blocks.


In [87]:
class CustomPreprocessor(Preprocessor):
    """Custom Preprocessor class."""
    def __init__(self) -> None:
        super().__init__()
    
    def _fit(self, ds):
        tags = ds.unique(column="tag")
        self.class_to_index = {tag: i for i, tag in enumerate(tags)}
        self.index_to_class = {i: tag for i, tag in enumerate(tags)}

    def _transform_pandas(self, batch):
        return preprocess(batch, class_to_index = self.class_to_index)

### Model

In [88]:
import torch.nn as nn
from transformers import BertModel

In [89]:
llm = BertModel.from_pretrained("allenai/scibert_scivocab_uncased", return_dict = False)
embedding_dim = llm.config.hidden_size
embedding_dim

/home/krekken/madewithml/venv/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at allenai/scibert_scivocab_uncased were not used when initializing BertModel: ['cls.seq_relationship.bias', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.weight', 'cls.predictions.bias', 'cls.predictions.decoder.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.decoder.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you a

768

In [90]:
# Tokenizer
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("allenai/scibert_scivocab_uncased")
text = "Single cell RNA-seq reveals distinct cell types."
tokens = tokenizer.tokenize(text)
input_ids = tokenizer.encode(text, return_tensors="pt")
input_ids

tensor([[  102,  1232,   377,  2980,   579, 26317,  8234,  3646,   377,  1910,
           205,   103]])

In [91]:
text = "Transfer learning with transformers for text classification."
batch = tokenizer([text], return_tensors = "pt", padding="longest")
batch

{'input_ids': tensor([[  102,  2268,  1904,   190, 29155,   168,  3267,  2998,   205,   103]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [92]:
batch["input_ids"]

tensor([[  102,  2268,  1904,   190, 29155,   168,  3267,  2998,   205,   103]])

In [93]:
seq,pool = llm(input_ids=batch["input_ids"], attention_mask = batch["attention_mask"])
seq.size(), pool.size()

(torch.Size([1, 10, 768]), torch.Size([1, 768]))

In [94]:
# Finetuning
class FinetunedLLM(nn.Module):
    def __init__(self, llm, dropout_p, embedding_dim, num_classes):
        super(FinetunedLLM, self).__init__()
        self.llm = llm
        self.dropout = nn.Dropout(dropout_p)
        self.fc1 = nn.Linear(embedding_dim, num_classes)

    def forward(self, batch):
        ids, masks = batch["ids"], batch["masks"]
        seq, pool = self.llm(input_ids = ids, attention_mask=masks)
        z = self.dropout(pool)
        z = self.fc1(z)
        return z
    @torch.inference_mode()
    def predict(self, batch):
        self.eval()
        z = self(batch)
        y_pred = torch.argmax(z, dim=1).cpu().numpy()
        return y_pred
    
    @torch.inference_mode()
    def predict_proba(self, batch):
        self.eval()
        z = self(batch)
        y_probs = F.softmax(z).cpu().numpy()
        return y_probs


In [95]:
# Initialize model
model = FinetunedLLM(llm = llm, dropout_p=0.5, embedding_dim=embedding_dim, num_classes=4)
model.named_parameters

<bound method Module.named_parameters of FinetunedLLM(
  (llm): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(31090, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): Layer

### Batching

In [96]:
from ray.train.torch import get_device

In [97]:
def pad_array(arr, dtype=np.int32):
    max_len = max(len(row) for row in arr)
    padded_arr = np.zeros((arr.shape[0], max_len), dtype=dtype)
    for i, row in enumerate(arr):
        padded_arr[i][:len(row)] = row
    return padded_arr

In [98]:
def collate_fn(batch):
    batch["id"] = pad_array(batch["id"])
    batch["masks"] = pad_array(batch["masks"])
    dtypes = {"id": torch.int32, "masks": torch.int32, "targets": torch.int64}
    tensor_batch = {}
    for key, array in batch.items():
        tensor_batch[key] = torch.as_tensor(array, dtype=dtypes[key], device=device)

In [99]:
# Sample batch
sample_batch = sample_ds.take_batch(batch_size=128)
# collate_fn(batch=sample_batch)

2026-02-26 20:39:13,280	INFO streaming_executor.py:93 -- Executing DAG InputDataBuffer[Input] -> TaskPoolMapOperator[ReadCSV->SplitBlocks(40)] -> AllToAllOperator[RandomShuffle] -> LimitOperator[limit=128]
2026-02-26 20:39:13,281	INFO streaming_executor.py:94 -- Execution config: ExecutionOptions(resource_limits=ExecutionResources(cpu=None, gpu=None, object_store_memory=None), locality_with_output=False, preserve_order=False, actor_locality_enabled=True, verbose_progress=False)
2026-02-26 20:39:13,281	INFO streaming_executor.py:96 -- Tip: For detailed progress reporting, run `ray.data.DataContext.get_current().execution_options.verbose_progress = True`


- RandomShuffle 1:   0%|          | 0/1600 [00:00<?, ?it/s]

Shuffle Map 2:   0%|          | 0/1600 [00:00<?, ?it/s]

Shuffle Reduce 3:   0%|          | 0/1600 [00:00<?, ?it/s]

Running 0:   0%|          | 0/1 [00:00<?, ?it/s]

In [100]:
for row in sample_batch["id"]:
    print(type(row))

<class 'numpy.int64'>
<class 'numpy.int64'>
<class 'numpy.int64'>
<class 'numpy.int64'>
<class 'numpy.int64'>
<class 'numpy.int64'>
<class 'numpy.int64'>
<class 'numpy.int64'>
<class 'numpy.int64'>
<class 'numpy.int64'>
<class 'numpy.int64'>
<class 'numpy.int64'>
<class 'numpy.int64'>
<class 'numpy.int64'>
<class 'numpy.int64'>
<class 'numpy.int64'>
<class 'numpy.int64'>
<class 'numpy.int64'>
<class 'numpy.int64'>
<class 'numpy.int64'>
<class 'numpy.int64'>
<class 'numpy.int64'>
<class 'numpy.int64'>
<class 'numpy.int64'>
<class 'numpy.int64'>
<class 'numpy.int64'>
<class 'numpy.int64'>
<class 'numpy.int64'>
<class 'numpy.int64'>
<class 'numpy.int64'>
<class 'numpy.int64'>
<class 'numpy.int64'>
<class 'numpy.int64'>
<class 'numpy.int64'>
<class 'numpy.int64'>
<class 'numpy.int64'>
<class 'numpy.int64'>
<class 'numpy.int64'>
<class 'numpy.int64'>
<class 'numpy.int64'>
<class 'numpy.int64'>
<class 'numpy.int64'>
<class 'numpy.int64'>
<class 'numpy.int64'>
<class 'numpy.int64'>
<class 'nu

In [101]:
sample_batch["id"]

array([ 941, 1800,  291,  448,  253, 1953,  788,  632,   45,  993, 1492,
       2173, 1104, 2295, 2163, 1576, 1275, 1093, 1859, 2434, 1309,  877,
       2013, 1646,  778, 1658,  667, 1902, 1768,  857,  835,  455, 1115,
       2117, 2006, 1473, 1239, 1165, 1084,  474, 2317, 1589,  883, 1899,
        800, 1597,  286,  372,  145, 1927,  469,  443,   29,  914, 2251,
       1943, 2135, 1258, 1721, 1026,  948, 2373, 1475,  687, 2226, 1730,
       2012, 2038, 2422,  159, 2235, 2356,   98,   77, 1083, 1130, 1260,
        902, 1802,  425, 2371, 1710, 1918, 1965,   78,  316, 2296, 1478,
        101, 1617, 1542,  516, 1868,  884, 1020, 2284, 2298, 2047,  120,
       2204, 2430,  480, 1994, 1722,  430, 1952, 1880,  847, 1457, 1519,
        688,  378, 1603,  280, 1818, 1636,  732,  702,  224,  974, 1919,
       1860, 1506, 1427, 1236,  890,  256, 2280])

In [102]:
sample_batch["tag"].shape

(128,)

### Utilities

In [103]:
from ray.air import Checkpoint, session
from ray.air.config import CheckpointConfig, DatasetConfig, RunConfig, ScalingConfig
import ray.train as train
from ray.train.torch import TorchCheckpoint, TorchTrainer
import torch.nn.functional as F

In [104]:
def train_step(ds, batch_size, model, num_classes, loss_fn, optimizer):
    """Train Step"""
    model.train()
    loss = 0.0
    ds_generator = ds.iter_torch_batches(batch_size=batch_size, collate_fn=collate_fn)
    for i, batch in enumerate(ds_generator):
        optimizer.zero_grad()
        z = model.forward(batch)
        targets = F.one_hot(batch["targets"], num_classes=num_classes).float()
        J = loss_fn(z, targets)
        optimizer.step()
        loss += (J.detach().item() - loss) / (i + 1)
    return loss

In [105]:
def eval_step(ds, model, batch_size, loss_fn, num_classes):
    model.eval()
    loss = 0.0
    y_trues, y_preds = [], []
    ds_generator = ds.iter_torch_batches(batch_size=batch_size, collate_fn=collate_fn)
    with torch.inference_mode():
        for i, batch in enumerate(ds_generator):
            z = model(batch)
            targets = F.one_hot(batch["targets"], num_classes=num_classes).float()
            J = loss_fn(z, targets).item()
            loss += (J - loss) / (i + 1)
            y_trues.extend(batch["targets"].cpu().numpy())
            y_preds.extend(batch["targets"].cpu().numpy())
            y_preds.extend(torch.argmax(z, dim=1).cpu().numpy())
    return loss, np.vstack(y_trues), np.vstack(y_preds)


In [106]:
# Training Loop
def train_loop_per_worker(config):
    #Hyperparameters
    dropout_p = config["dropout_p"]
    lr = config["lr"]
    lr_factor = config["lr_factor"]
    lr_patience = config["lr_patience"]
    num_epochs = config["num_epochs"]
    batch_size = config["batch_size"]
    num_classes = config["num_classes"]

    # Get datasets
    set_seeds()
    train_ds = session.get_dataset_shard("train")
    val_ds = session.get_dataset_shard("val")

    #Model
    llm = BertModel.from_pretrained("allenai/scibert_scivocab_uncased", return_dict=False)
    model = FinetunedLLM(llm, dropout_p, llm.config.hidden_size, num_classes)
    model = train.torch.prepare_model(model)

    #Training Components
    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer=optimizer, mode="min", factor=lr_factor, patience=lr_patience)

    #training
    batch_size_per_worker = batch_size // session.get_world_size()
    for epoch in range(num_epochs):
        #step
        train_loss = train_step(train_ds, batch_size_per_worker, model, num_classes, loss_fn, optimizer)
        val_loss, _, _ = eval_step(val_ds, model, batch_size, loss_fn, num_classes)
        scheduler.step(val_loss)

        #chekpoint
        metrics = dict(epoch=epoch, lr=optimizer.param_groups[0]["lr"], train_loss=train_loss, val_loss=val_loss)
        checkpoint = TorchCheckpoint.from_model(model=model)
        session.report(metrics, checkpoint)

In [107]:
num_classes = len(tags)
num_classes

4

### Configurations

In [108]:
#Train loop config
train_loop_config = {
    "dropout_p": 0.5,
    "lr": 1e-4,
    "lr_factor": 0.8,
    "lr_patience": 3,
    "num_epochs": 10,
    "batch_size": 256,
    "num_classes": num_classes
    }

In [109]:
#Scaling config
preproccess_scaling_config = ScalingConfig(
    num_workers = 15,
    use_gpu=False,
    resources_per_worker={
        "CPU": 1,
        "GPU": 0
    }
)
finetune_scaling_config = ScalingConfig(
    num_workers=1,
    use_gpu=True,
    resources_per_worker={
        "CPU": 1,
        "GPU": 1
    }
)

In [110]:
#Run config
checkpoint_config = CheckpointConfig(num_to_keep=1, checkpoint_score_attribute="val_loss", checkpoint_score_order="min")
run_config = RunConfig(name="llm", checkpoint_config=checkpoint_config, local_dir="~/ray_results")

### Training

In [111]:
import sys
sys.path.append("..")
from madewithml.data import stratify_split

In [112]:
#Load and split data
ds = load_data()
test_size = 0.2
train_ds, val_ds = stratify_split(ds, stratify="tag", test_size=test_size)

2026-02-26 20:39:16,748	INFO read_api.py:406 -- To satisfy the requested parallelism of 40, each read task output is split into 40 smaller blocks.
2026-02-26 20:39:16,758	INFO streaming_executor.py:93 -- Executing DAG InputDataBuffer[Input] -> TaskPoolMapOperator[ReadCSV->SplitBlocks(40)] -> AllToAllOperator[RandomShuffle] -> LimitOperator[limit=1]
2026-02-26 20:39:16,760	INFO streaming_executor.py:94 -- Execution config: ExecutionOptions(resource_limits=ExecutionResources(cpu=None, gpu=None, object_store_memory=None), locality_with_output=False, preserve_order=False, actor_locality_enabled=True, verbose_progress=False)
2026-02-26 20:39:16,761	INFO streaming_executor.py:96 -- Tip: For detailed progress reporting, run `ray.data.DataContext.get_current().execution_options.verbose_progress = True`


- RandomShuffle 1:   0%|          | 0/1600 [00:00<?, ?it/s]

Shuffle Map 2:   0%|          | 0/1600 [00:00<?, ?it/s]

Shuffle Reduce 3:   0%|          | 0/1600 [00:00<?, ?it/s]

Running 0:   0%|          | 0/1 [00:00<?, ?it/s]

In [113]:
ds

RandomShuffle
+- Dataset(
      num_blocks=40,
      num_rows=764,
      schema={
         id: int64,
         created_on: timestamp[s],
         title: string,
         description: string,
         tag: string
      }
   )

In [114]:
from transformers import BertTokenizer
import re
from wordcloud import STOPWORDS
def clean_text(text, stopwords=STOPWORDS):
    """Clean raw text string."""
    # Lower
    text = text.lower()

    # Remove stopwords
    pattern = re.compile(r'\b(' + r"|".join(stopwords) + r")\b\s*")
    text = pattern.sub('', text)

    # Spacing and filters
    text = re.sub(r"([!\"'#$%&()*\+,-./:;<=>?@\\\[\]^_`{|}~])", r" \1 ", text)  # add spacing
    text = re.sub("[^A-Za-z0-9]+", " ", text)  # remove non alphanumeric chars
    text = re.sub(" +", " ", text)  # remove multiple spaces
    text = text.strip()  # strip white space at the ends
    text = re.sub(r"http\S+", "", text)  #  remove links
    
    return text


def tokenize(batch):
    tokenizer = BertTokenizer.from_pretrained("allenai/scibert_scivocab_uncased", return_dict=False)
    encoded_inputs = tokenizer(batch["text"].tolist(), padding="longest", return_tensors="np")
    return dict(ids=encoded_inputs["input_ids"], masks=encoded_inputs["attention_mask"], tag=np.array(batch["tag"]))

def preprocess(df, class_to_index):
    df["text"] = df["title"] + " " + df["description"]
    df["text"] = df["text"].apply(clean_text)
    df = df.drop(columns=["title", "id", "created_on", "description"])
    df = df[["text", "tag"]]
    df["tag"] = df["tag"].map(class_to_index)
    outputs = tokenize(df)

    return outputs

In [115]:
train_ds

RandomShuffle
+- MapBatches(_filter_split)
   +- MapBatches(group_fn)
      +- Sort
         +- RandomShuffle
            +- Dataset(
                  num_blocks=40,
                  num_rows=764,
                  schema={
                     id: int64,
                     created_on: timestamp[s],
                     title: string,
                     description: string,
                     tag: string
                  }
               )

In [116]:
#Preprocess
preprocessor = CustomPreprocessor()
train_ds = preprocessor.fit_transform(train_ds)
val_ds = preprocessor.transform(val_ds)


2026-02-26 20:39:23,388	INFO streaming_executor.py:93 -- Executing DAG InputDataBuffer[Input] -> TaskPoolMapOperator[ReadCSV->SplitBlocks(40)] -> AllToAllOperator[RandomShuffle] -> AllToAllOperator[Sort] -> AllToAllOperator[MapBatches(group_fn)->MapBatches(_filter_split)->RandomShuffle] -> LimitOperator[limit=1]
2026-02-26 20:39:23,389	INFO streaming_executor.py:94 -- Execution config: ExecutionOptions(resource_limits=ExecutionResources(cpu=None, gpu=None, object_store_memory=None), locality_with_output=False, preserve_order=True, actor_locality_enabled=True, verbose_progress=False)
2026-02-26 20:39:23,389	INFO streaming_executor.py:96 -- Tip: For detailed progress reporting, run `ray.data.DataContext.get_current().execution_options.verbose_progress = True`


- RandomShuffle 1:   0%|          | 0/1600 [00:00<?, ?it/s]

Shuffle Map 2:   0%|          | 0/1600 [00:00<?, ?it/s]

Shuffle Reduce 3:   0%|          | 0/1600 [00:00<?, ?it/s]

- Sort 4:   0%|          | 0/1600 [00:00<?, ?it/s]

...

Shuffle Map 12:   0%|          | 0/1600 [00:00<?, ?it/s]

Shuffle Reduce 13:   0%|          | 0/1600 [00:00<?, ?it/s]

Running 0:   0%|          | 0/1600 [00:00<?, ?it/s]

Sort Sample 0:   0%|          | 0/40 [00:00<?, ?it/s]

Sort Sample 0:   0%|          | 0/40 [00:00<?, ?it/s]